# Task 2 — tuned CatBoost plus class weighting (v7)

This one focused follow-up applies the previously tested Social/Whale weights `[1, 1, 1.25, 1.25]` to the v6 development-fold winner. It reads the saved `eda_original_v2` card and v6 model card, reproduces the same five development folds, and evaluates the distinct saved model once on the **reused** 20% holdout. The tier timing remains unverified. No Test rows enter training or tuning.

In [1]:
from pathlib import Path
import sys, json, hashlib, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
from catboost import CatBoostClassifier
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task2.models.decision_rules import restore_categorical_nulls
SEED=set_seed()
DATA='eda_original_v2'; BASE='t2_cat_tuned_cv5_v6'; NAME='t2_cat_tuned_weighted_cv5_v7'
data_card=json.loads((SRC/'task2'/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
base_card=json.loads((SRC/'task2'/'save'/f'{BASE}.json').read_text(encoding='utf-8'))
train=pd.read_parquet(SRC/data_card['splits']['train']['path'])
y=train.segment.astype(int)
raw=train.drop(columns=['player_id','segment'])
cats=raw.select_dtypes(include=['object']).columns.tolist()
X=restore_categorical_nulls(raw,cats)
dev_pos,hold_pos=train_test_split(np.arange(len(y)),test_size=.20,stratify=y,random_state=SEED)
X_dev,X_hold=X.iloc[dev_pos].reset_index(drop=True),X.iloc[hold_pos]
y_dev,y_hold=y.iloc[dev_pos].reset_index(drop=True),y.iloc[hold_pos]
folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(X_dev,y_dev))
source=joblib.load(SRC/base_card['model_path'])
pre=source.named_steps['preprocess']
params=source.named_steps['classifier'].get_params()
assert params['random_seed']==SEED and params.get('class_weights') is None
weights=[1.0,1.0,1.25,1.25]
def make_model():
    return Pipeline([('preprocess',clone(pre)),
                     ('classifier',CatBoostClassifier(**{**params,'class_weights':weights}))])
OUT=SRC/'task2'/'models'/'results'; OUT.mkdir(parents=True,exist_ok=True)
cache=OUT/'t2_cat_tuned_weighted_cv5_v7.json'
key={'data':DATA,'data_created_at':data_card['created_at'],'base':BASE,
     'base_created_at':base_card['created_at'],'weights':weights,'seed':SEED,
     'split':hashlib.sha256(np.asarray(dev_pos,dtype=np.int64).tobytes()+np.asarray(y_dev,dtype=np.int8).tobytes()).hexdigest()}
result=json.loads(cache.read_text(encoding='utf-8')) if cache.exists() else {'key':key,'rows':[]}
assert result['key']==key
lock=threading.Lock()
def fit_fold(i,fit_pos,check_pos):
    if any(r['fold']==i for r in result['rows']):return
    t=time.monotonic(); model=make_model()
    model.fit(X_dev.iloc[fit_pos],y_dev.iloc[fit_pos])
    pred=np.asarray(model.predict(X_dev.iloc[check_pos])).reshape(-1).astype(int)
    row={'fold':i,'macro_F1':float(f1_score(y_dev.iloc[check_pos],pred,average='macro')),
         'seconds':round(time.monotonic()-t,1)}
    with lock:
        result['rows'].append(row)
        cache.write_text(json.dumps(result,indent=2)+'\n',encoding='utf-8')
    print(row,flush=True)
pending=[(i,a,b) for i,(a,b) in enumerate(folds,1) if not any(r['fold']==i for r in result['rows'])]
with ThreadPoolExecutor(max_workers=3) as pool:
    jobs=[pool.submit(fit_fold,i,a,b) for i,a,b in pending]
    for job in as_completed(jobs):job.result()
assert len(result['rows'])==5
weighted=pd.DataFrame(result['rows']).set_index('fold').sort_index().macro_F1
prior=json.loads((OUT/'t2_cat_finalists_v6.json').read_text(encoding='utf-8'))
v6=pd.DataFrame(prior['rows']).query('label == "search_01"').set_index('fold').sort_index().score
assert weighted.index.equals(v6.index)
paired=pd.DataFrame({'tuned_v6':v6,'tuned_weighted_v7':weighted,'delta':weighted-v6})
display(paired.round(6))
print('Mean v6:',v6.mean(),'mean weighted:',weighted.mean(),
      'delta:',paired.delta.mean(),'wins:',int((paired.delta>0).sum()),'/5')

{'fold': 1, 'macro_F1': 0.7566319668477723, 'seconds': 269.4}


{'fold': 2, 'macro_F1': 0.7501714091881371, 'seconds': 271.0}


{'fold': 3, 'macro_F1': 0.752881682182914, 'seconds': 271.0}


{'fold': 4, 'macro_F1': 0.7578678468425681, 'seconds': 181.9}


{'fold': 5, 'macro_F1': 0.7532286072602102, 'seconds': 180.9}


,tuned_v6,tuned_weighted_v7,delta
fold,,,
1,0.758175,0.756632,-0.001543
2,0.749225,0.750171,0.000946
3,0.753641,0.752882,-0.000759
4,0.756726,0.757868,0.001142
5,0.752707,0.753229,0.000522


Mean v6: 0.754094868982061 mean weighted: 0.7541563024643203 delta: 6.143348225939871e-05 wins: 3 /5


In [2]:
# Distinct fitted model, save, and holdout evaluation.
SAVE=SRC/'task2'/'save'
paths=[SAVE/f'{NAME}.joblib',SAVE/f'{NAME}.json',SAVE/f'{NAME}-evaluation.json']
if not any(p.exists() for p in paths):
    model=make_model(); model.fit(X_dev,y_dev)
    pred=np.asarray(model.predict(X_hold)).reshape(-1).astype(int)
    save_model(task=2,name=NAME,model=model,data_name=DATA,
        card={'idea':'Previously tested Social/Whale weights on v6 tuned CatBoost',
              'base_model':BASE,'weights':weights,'seed':SEED,
              'mean_cv_macro_F1':float(weighted.mean()),
              'mean_paired_cv_gain':float(paired.delta.mean()),
              'tier_timing':'unverified'})
    evaluation=evaluate_model(task=2,name=NAME,y_true=y_hold,y_pred=pred,
        card={'split':'reused stratified 20% holdout','seed':SEED,
              'selection':'fixed prior class weights on v6 tuned config'})
else:
    assert all(p.exists() for p in paths),'Inspect incomplete artifact'
    saved=json.loads(paths[1].read_text(encoding='utf-8'))
    assert saved['data_name']==DATA and saved['card']['base_model']==BASE
    model=joblib.load(paths[0]); pred=np.asarray(model.predict(X_hold)).reshape(-1).astype(int)
    evaluation=json.loads(paths[2].read_text(encoding='utf-8'))
v6_pred=np.asarray(source.predict(X_hold)).reshape(-1).astype(int)
print('Reused holdout v6:',f1_score(y_hold,v6_pred,average='macro'))
print('Reused holdout v7:',evaluation['score'])
print('Changed predictions:',int((v6_pred!=pred).sum()))
print(classification_report(y_hold,pred,digits=5))

Reused holdout v6: 0.7510954107114675
Reused holdout v7: 0.7528947062270108
Changed predictions: 601
              precision    recall  f1-score   support

           0    0.79258   0.84483   0.81787      8191
           1    0.75613   0.73420   0.74501      5207
           2    0.72336   0.69567   0.70925      4206
           3    0.76786   0.71308   0.73945      3196

    accuracy                        0.76673     20800
   macro avg    0.75998   0.74695   0.75289     20800
weighted avg    0.76566   0.76673   0.76562     20800

